## Give Claude custom tools

In [1]:
from typing import Any
import httpx
from claude_agent_sdk import (
    tool,
    create_sdk_mcp_server,
    query,
    ClaudeAgentOptions,
    ResultMessage,
)


# Define a tool: name, description, input schema, handler
@tool(
    "get_temperature",
    "Get the current temperature at a location",
    {"latitude": float, "longitude": float},
)
async def get_temperature(args: dict[str, Any]) -> dict[str, Any]:
    async with httpx.AsyncClient() as client:
        response = await client.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": args["latitude"],
                "longitude": args["longitude"],
                "current": "temperature_2m",
                "temperature_unit": "fahrenheit",
            },
        )
        data = response.json()

    # Return a content array - Claude sees this as the tool result
    return {
        "content": [
            {
                "type": "text",
                "text": f"Temperature: {data['current']['temperature_2m']}°F",
            }
        ]
    }


# Wrap the tool in an in-process MCP server
weather_server = create_sdk_mcp_server(
    name="weather",
    version="1.0.0",
    tools=[get_temperature],
)


async def main():
    options = ClaudeAgentOptions(
        mcp_servers={"weather": weather_server},
        allowed_tools=["mcp__weather__get_temperature"],
    )

    async for message in query(
        prompt="What's the temperature in San Francisco?",
        options=options,
    ):
        # ResultMessage is the final message after all tool calls complete
        if isinstance(message, ResultMessage) and message.subtype == "success":
            print(message.result)


await main()

旧金山当前温度 **60.2°F（约 15.7°C）**。


In [4]:
from claude_agent_sdk import (
    query,
    tool,
    create_sdk_mcp_server,
    ToolAnnotations,
    ClaudeAgentOptions,
)
from typing import Any


@tool("test", "Just for testing", {}, annotations=ToolAnnotations(readOnlyHint=False))
async def foo(args: dict[str, Any]):
    return {"content": [{"type": "text", "text": "Hello, world!"}]}


mcp_servers = create_sdk_mcp_server(name="test", version="0.0.1", tools=[foo])


async def main():
    options = ClaudeAgentOptions(
        mcp_servers={"test": mcp_servers}, allowed_tools=["mcp__test__test"]
    )
    async for message in query(
        prompt="测试, 使用 mcp__test__test 工具",
        options=options,
    ):
        print(message)


await main()

HookEventMessage(subtype='hook_started', data={'type': 'system', 'subtype': 'hook_started', 'hook_id': '199e26f9-3b9f-4e34-b8aa-d6ed9773b269', 'hook_name': 'SessionStart:startup', 'hook_event': 'SessionStart', 'uuid': '39310f31-3339-473d-8b61-af36d4ba334d', 'session_id': 'd9832ccf-46f4-4a44-807c-f95b5e7cd65d'}, hook_event_name='SessionStart', session_id='d9832ccf-46f4-4a44-807c-f95b5e7cd65d', uuid='39310f31-3339-473d-8b61-af36d4ba334d')
HookEventMessage(subtype='hook_response', data={'type': 'system', 'subtype': 'hook_response', 'hook_id': '199e26f9-3b9f-4e34-b8aa-d6ed9773b269', 'hook_name': 'SessionStart:startup', 'hook_event': 'SessionStart', 'output': '', 'stdout': '', 'stderr': '', 'exit_code': 0, 'outcome': 'success', 'uuid': '0ce5d02c-cb33-4389-af16-8984db482e63', 'session_id': 'd9832ccf-46f4-4a44-807c-f95b5e7cd65d'}, hook_event_name='SessionStart', session_id='d9832ccf-46f4-4a44-807c-f95b5e7cd65d', uuid='0ce5d02c-cb33-4389-af16-8984db482e63')
SystemMessage(subtype='init', data={